In [1]:
!pip install -q langchain langchain-community langchain-text-splitters
!pip install -q langchain-groq langchain-huggingface
!pip install -q faiss-cpu sentence-transformers
!pip install -q pymupdf gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 18.7 MB/s eta 0:00:00


In [2]:
import os
import tempfile
import gradio as gr

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

/tmp/ipykernel_5067/1244662037.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [4]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("Please configure GROQ_API_KEY in Colab Secrets.")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [10]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    api_key=GROQ_API_KEY
)

print("AI models initialized successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

AI models initialized successfully!


In [11]:
response = llm.invoke("Explain what RAG means in one sentence.")
print(response.content)

RAG stands for Retrieval‑Augmented Generation, a technique that blends a language model with an external knowledge‑retrieval system to produce more accurate and context‑aware responses.


In [6]:
def process_pdf(pdf_path):

    loader = PyMuPDFLoader(pdf_path)
    documents = loader.load()

    if not documents:
        raise ValueError("No readable text found in PDF.")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150
    )

    chunks = splitter.split_documents(documents)

    if not chunks:
        raise ValueError("Unable to create document chunks.")

    vectorstore = FAISS.from_documents(
        chunks,
        embeddings
    )

    return vectorstore, len(chunks)

In [7]:
prompt = ChatPromptTemplate.from_template("""
You are an intelligent AI Document Assistant.

Answer the user's question using only the provided context.

Instructions:
1. Give accurate and clear answers.
2. Do not invent information.
3. If the answer is not available in the document, say:
   "I could not find this information in the uploaded document."
4. Use bullet points when helpful.
5. Keep the response professional.

Context:
{context}

Question:
{question}

Answer:
""")

In [8]:
def ask_document(question, vectorstore):

    if not question.strip():
        return "Please enter a question.", ""

    docs = vectorstore.similarity_search(
        question,
        k=4
    )

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    chain = prompt | llm

    response = chain.invoke({
        "context": context,
        "question": question
    })

    answer = response.content

    sources = []

    for doc in docs:
        page = doc.metadata.get("page", 0) + 1
        sources.append(f"Page {page}")

    source_text = ", ".join(sorted(set(sources)))

    return answer, source_text

In [9]:
vectorstore = None

def upload_document(file):

    global vectorstore

    if file is None:
        return "Please upload a PDF first."

    try:
        vectorstore, chunks = process_pdf(file.name)

        return (
            f"Document processed successfully!\n"
            f"Total text chunks: {chunks}\n"
            f"You can now ask questions."
        )

    except Exception as e:
        vectorstore = None
        return f"Error processing PDF: {str(e)}"


def chat_with_document(question, history):

    global vectorstore

    if vectorstore is None:
        return "Please upload a PDF document first."

    try:
        answer, sources = ask_document(
            question,
            vectorstore
        )

        return f"{answer}\n\n**Sources:** {sources}"

    except Exception as e:
        return f"Error: {str(e)}"


with gr.Blocks(
    theme=gr.themes.Soft(),
    title="AI Document Assistant"
) as demo:

    gr.Markdown("""
    # AI Document Assistant

    ### Ask questions and get intelligent answers from your PDF documents.

    Upload a document, process it, and start chatting.
    """)

    with gr.Row():

        pdf_file = gr.File(
            label="Upload PDF",
            file_types=[".pdf"],
            type="filepath"
        )

        upload_status = gr.Textbox(
            label="Document Status",
            interactive=False
        )

    upload_button = gr.Button(
        "Process Document",
        variant="primary"
    )

    upload_button.click(
        upload_document,
        inputs=pdf_file,
        outputs=upload_status
    )

    chatbot = gr.ChatInterface(
        fn=chat_with_document,
        chatbot=gr.Chatbot(
            height=450,
            placeholder="Ask something about your document..."
        ),
        textbox=gr.Textbox(
            placeholder="Ask a question about your PDF...",
            container=False
        )
    )

demo.launch(share=True)

/tmp/ipykernel_5067/148249520.py:43: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e5e4c1e7da6688eb8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
